## Circular Task Analysis

#Background

This project analyzes data from the circular task described in Fauviaux et al. (2021). Participants move the mouse along a ring as fast and as accurately as possible. The data were recorded with the MouseReMoCo software.

The session consists of 5 recordings of 20 s, separated by 20 s pauses.



# Code to help us

In [ ]:
# imports (this is all you need for this assignment - please, do not import anything else)
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle , Circle
import os

# very useful for plots that yo can zoom in-out
%matplotlib widget

# import pandas as pd # we do not use pandas here
# if the present working directory is the directory of this file, change the path to the project root
if os.getcwd().endswith("notebooks"):
    os.chdir("..")

## Header Information

In [6]:
DATA_FILE = "data/001MoDe_R1.csv"

# Read the header line (first line: "key value;key value;...")
with open(DATA_FILE, "r", encoding="utf-8") as f:
    first_line = f.readline().strip()

header = {}
for item in first_line.split(";"):
    key, _, value = item.partition(" ")
    try:
        header[key] = float(value)
    except ValueError:
        header[key] = value

# Display the header information
print(f"Header line information in '{DATA_FILE}':")
for key, value in header.items():
    print(f"  {key}: {value}")

Header line information in 'data/001MoDe_R1.csv':
  screenWidth: 1104.0
  screenHeight: 661.0
  cornerX: 302.0
  cornerY: 80.0
  centerX: 552.0
  centerY: 330.0
  externalRadius: 250.0
  internalRadius: 170.0
  borderRadius: 1.0
  cursorRadius: 16.0
  indexOfDifficulty: 28.00696429476858
  taskRadius: 209.5
  taskTolerance: 47.0
  borderColor: java.awt.Color[r=255,g=255,b=255]
  backgroundColor: java.awt.Color[r=0,g=0,b=0]
  cursorColorRecord: java.awt.Color[r=255,g=0,b=0]
  cursorColorWait: java.awt.Color[r=255,g=255,b=0]
  autoStart: 3600.0
  cycleMaxNumber: 6.0
  cycleDuration: 20.0
  software: LSL-mouse
  version: 1.2.0rc5
  task: CircularTarget
  isWithLSL: false


## Markers Information

In [7]:
MARKER_FILE = "data/001MoDe_R1.marker.csv"

# Read the markers: each line is "date,timestamp,event"
markers = []
with open(MARKER_FILE, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split(",", 2)
        if len(parts) == 3 and parts[1].isdigit():
            markers.append((int(parts[1]), parts[2]))

# Display the markers
print(f"Markers in '{MARKER_FILE}':")
for timestamp, event in markers:
    print(f"{timestamp}: {event}")

Markers in 'data/001MoDe_R1.marker.csv':
1616776712325: KeyTyped=32 DoCycleChange
1616776712331: DoCycleChange:DoStartCycleTimedSequence
1616776712333: DoCycleChange:DoRecord RecordDone=0 PauseDone=0 ToDo=6
1616776732336: DoCycleChange:DoPause RecordDone=1 PauseDone=0 ToDo=6
1616776752340: DoCycleChange:DoRecord RecordDone=1 PauseDone=1 ToDo=6
1616776772351: DoCycleChange:DoPause RecordDone=2 PauseDone=1 ToDo=6
1616776792372: DoCycleChange:DoRecord RecordDone=2 PauseDone=2 ToDo=6
1616776812386: DoCycleChange:DoPause RecordDone=3 PauseDone=2 ToDo=6
1616776832390: DoCycleChange:DoRecord RecordDone=3 PauseDone=3 ToDo=6
1616776852404: DoCycleChange:DoPause RecordDone=4 PauseDone=3 ToDo=6
1616776872419: DoCycleChange:DoRecord RecordDone=4 PauseDone=4 ToDo=6
1616776892433: DoCycleChange:DoPause RecordDone=5 PauseDone=4 ToDo=6
1616776898364: KeyTyped=113 WINDOW_CLOSING
1616776898367: 


## Software Summary Table

At the end of `001MoDe_R1.marker.csv`, MouseReMoCo writes a summary table with the statistics it computed for each record (`nLaps`, `Re`, `Te`, `error`, `MT/lap`, `IDe/lap`, `Be`, `IPe`). The first row ("Theory") gives the values expected from the task parameters. We will later recompute these values from the raw data and compare them with this table.

In [10]:
MARKER_FILE = "data/001MoDe_R1.marker.csv"

# Print the summary table computed by the software (end of the marker file)
print(f"Software summary table in '{MARKER_FILE}':")
with open(MARKER_FILE, "r", encoding="utf-8") as f:
    table_started = False
    for line in f:
        line = line.strip()
        if "Var" in line and "nLaps" in line:   # start of the table
            table_started = True
        if table_started:
            print(line)

Software summary table in 'data/001MoDe_R1.marker.csv':
Var , nLaps ,      Re ,      Te ,   error ,  MT/lap , IDe/lap ,      Be ,     IPe ,
unit ,   lap ,   pixel ,   pixel ,       % ,   s/lap , bit/lap ,  double ,   bit/s ,
Theory ,  1.00 ,  209.50 ,   47.00 ,    3.88 ,         ,         ,    1.00 ,         ,
Rec001 ,-10.85 ,  213.87 ,   91.79 ,    3.90 ,    1.84 ,   14.64 ,    1.95 ,    7.94 ,
Rec002 ,-11.97 ,  209.85 ,   34.46 ,    0.74 ,    1.59 ,   38.27 ,    0.73 ,   24.04 ,
Rec003 ,-13.26 ,  209.43 ,   42.42 ,    2.81 ,    1.45 ,   31.02 ,    0.90 ,   21.40 ,
Rec004 ,-12.83 ,  209.46 ,   37.14 ,    0.13 ,    1.49 ,   35.43 ,    0.79 ,   23.72 ,
Rec005 ,-14.74 ,  210.89 ,   43.88 ,    2.84 ,    1.30 ,   30.20 ,    0.93 ,   23.20 ,
Rec006 ,  0.00 ,    0.00 ,    0.00 ,     NaN ,     NaN ,     NaN ,    0.00 ,    0.00 ,
